# Bundestagswahl 2021: Wahlkreisdaten Schritt für Schritt aufbereiten

Dieses Notebook zeigt jeden Verarbeitungsschritt offen. Die eigentliche Rechenlogik liegt in kleinen Python-Funktionen, aber keine große Pipeline wird als Blackbox aufgerufen.

Die Wahlbezirksdatei wird zuerst geprüft und anschließend auf Wahlkreise zusammengefasst. Die repräsentative Statistik liefert nur Verteilungen auf Landesebene. Ihre gerundeten absoluten Werte werden deshalb in Anteile umgerechnet und später auf die amtlichen Stimmen aus der Wahlbezirksdatei angewendet.

Die Quellkategorie `m` enthält laut Hinweis der Bundeswahlleitung auch Personen mit dem Geschlechtsmerkmal divers sowie Personen ohne Geschlechtseintrag im Geburtenregister.

## 0. Arbeitsverzeichnis und lokale Dateien

In [ ]:
from pathlib import Path
import sys
from dataclasses import asdict

import pandas as pd
from IPython.display import display


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts").is_dir() and (candidate / "package.json").is_file():
            return candidate
    raise RuntimeError("Das Notebook muss innerhalb des Repository-Ordners laufen.")


ROOT = find_repository_root()
sys.path.insert(0, str(ROOT))
ROOT

In [ ]:
# Diese Namen sind nur lokale Vorschläge. Trage hier die tatsächlich
# heruntergeladenen CSV-Dateien ein.
DISTRICT_RESULTS_CSV = ROOT / "scripts/data/btw21_wbz_ergebnisse.csv"
STATE_DEMOGRAPHICS_CSV = ROOT / "scripts/data/btw21_rws_stimmabgabe_laender.csv"
FEDERAL_METHOD_DEMOGRAPHICS_CSV = None  # optionaler lokaler Pfad
OUTPUT_DIRECTORY = ROOT / "scripts/data/generated"

DISTRICT_RESULTS_CSV, STATE_DEMOGRAPHICS_CSV, FEDERAL_METHOD_DEMOGRAPHICS_CSV

In [ ]:
from scripts.election_data.btw2021 import read_local_csv
from scripts.election_data.notebook_steps import (
    aggregate_to_constituencies,
    calculate_demographic_profiles,
    calculate_federal_method_weights,
    district_party_columns,
    inspect_district_rows,
    normalize_district_rows,
    normalize_federal_method_rows,
    normalize_state_statistic_rows,
    reshape_federal_method_votes,
    reshape_polling_district_votes,
    reshape_state_statistic_votes,
    select_federal_method_detail_rows,
    select_state_statistic_detail_rows,
    select_usable_district_rows,
)
from scripts.election_data.pipeline import distribute_district_votes, write_vote_entries
from scripts.election_data.profiles import build_state_method_profiles
from scripts.election_data.validation import validate_vote_entries

## 1. Wahlbezirksdatei unverändert einlesen

Zuerst wird nur gelesen. Noch wird keine Zeile entfernt und keine Spalte umbenannt.

In [ ]:
raw_districts = read_local_csv(DISTRICT_RESULTS_CSV)
print(f"Zeilen: {len(raw_districts):,}")
print(f"Spalten: {len(raw_districts.columns):,}")
display(raw_districts.head())
display(raw_districts.tail())

## 2. Wahlkreisnummern prüfen

Bevor `Wahlkreis` in eine Ganzzahl umgewandelt wird, werden alle Zeilen eingeteilt:

- `usable`: enthält eine normale Wahlkreisnummer;
- `missing`: leerer Wert oder eine leere Abschlusszeile;
- `invalid`: nicht leer, aber keine Zahl. Solche Zeilen müssen untersucht werden.

Dadurch ist bei einem Fehler direkt sichtbar, welche Quellzeilen ihn verursachen.

In [ ]:
district_row_diagnostics = inspect_district_rows(raw_districts)
display(district_row_diagnostics["status"].value_counts().rename("rows"))

problematic_district_rows = district_row_diagnostics[
    district_row_diagnostics["status"] != "usable"
]
display(problematic_district_rows)

Jetzt werden nur die zuvor sichtbaren leeren Zeilen entfernt. Nicht leere, ungültige Werte führen weiterhin zu einem Fehler mit den konkreten Quellzeilen.

In [ ]:
usable_district_rows = select_usable_district_rows(
    raw_districts,
    district_row_diagnostics,
)
print(f"Verwendete Zeilen: {len(usable_district_rows):,}")
display(usable_district_rows.head())

## 3. Wahlkreis, Bundesland und Wahlart vereinheitlichen

Die Wahlkreisnummer wird jetzt zu `districtId`. Die Landeskennziffer wird zum ausgeschriebenen Bundesland. Die Bezirksarten werden für die App auf `postal` und `in-person` abgebildet.

In [ ]:
normalized_district_rows = normalize_district_rows(usable_district_rows)

display(
    normalized_district_rows[
        ["Wahlkreis", "districtId", "Land", "state", "Bezirksart", "electionMethod"]
    ].head(20)
)
display(normalized_district_rows["electionMethod"].value_counts())

## 4. Parteienfelder der Erst- und Zweitstimmen bestimmen

Die Spalten `E_Gültige`, `E_Ungültige`, `Z_Gültige` und `Z_Ungültige` sind Summen und keine Parteien. Alle übrigen `E_`- beziehungsweise `Z_`-Spalten bleiben erhalten.

In [ ]:
first_vote_columns = district_party_columns(normalized_district_rows, "E_")
second_vote_columns = district_party_columns(normalized_district_rows, "Z_")

print(f"Erststimmen-Parteifelder: {len(first_vote_columns)}")
print(f"Zweitstimmen-Parteifelder: {len(second_vote_columns)}")
display(pd.Series(first_vote_columns, name="first vote columns").to_frame())
display(pd.Series(second_vote_columns, name="second vote columns").to_frame())

## 5. Breite Parteispalten in einzelne Stimmenzeilen umformen

Jede Zeile beschreibt danach genau eine Kombination aus Quellzeile, Wahlkreis, Partei, Stimmenart und Brief-/Urnenwahl. Zu diesem Zeitpunkt liegen die Daten weiterhin auf Wahlbezirksebene vor.

In [ ]:
first_polling_district_votes = reshape_polling_district_votes(
    normalized_district_rows,
    prefix="E_",
    vote_type="1",
)
second_polling_district_votes = reshape_polling_district_votes(
    normalized_district_rows,
    prefix="Z_",
    vote_type="2",
)

display(first_polling_district_votes.head(20))
display(second_polling_district_votes.head(20))

## 6. Wahlbezirke zu Wahlkreisen zusammenfassen

Für die App wird nicht jeder Wahlbezirk gespeichert. Alle Wahlbezirke desselben Wahlkreises werden nach Partei, Stimmenart und Brief-/Urnenwahl addiert.

In [ ]:
polling_district_votes = pd.concat(
    [first_polling_district_votes, second_polling_district_votes],
    ignore_index=True,
)
district_totals = aggregate_to_constituencies(polling_district_votes)

print(f"Wahlkreise: {district_totals['districtId'].nunique()}")
print(f"Bundesländer: {district_totals['state'].nunique()}")
print(f"Parteien/Sammelkategorien: {district_totals['party'].nunique()}")
print(f"Wahlkreis-Summen-Zeilen: {len(district_totals):,}")
display(district_totals.head(30))

In [ ]:
sample_district_id = int(district_totals["districtId"].min())
display(
    district_totals[district_totals["districtId"] == sample_district_id]
    .sort_values(["voteType", "electionMethod", "votes"], ascending=[True, True, False])
    .head(40)
)

## 7. Repräsentative Landesstatistik unverändert einlesen

Die Kommentarzeilen am Anfang beginnen mit `#` und werden beim Einlesen übersprungen. Die eigentliche Tabelle bleibt ansonsten unverändert.

In [ ]:
raw_state_statistics = read_local_csv(STATE_DEMOGRAPHICS_CSV, comment="#")
print(f"Zeilen: {len(raw_state_statistics):,}")
print(f"Spalten: {len(raw_state_statistics.columns):,}")
display(raw_state_statistics.head(20))

## 8. Dimensionen der Statistik vereinheitlichen

Bundesland, Erst-/Zweitstimme, Geschlecht und Altersgruppe erhalten die Werte, die später auch im JSON stehen. Summenzeilen bleiben zunächst erhalten und sind an leeren normalisierten Dimensionen erkennbar.

In [ ]:
normalized_state_statistics = normalize_state_statistic_rows(raw_state_statistics)

display(
    normalized_state_statistics[
        [
            "Land",
            "state",
            "Erst-/Zweitstimme",
            "voteType",
            "Geschlecht",
            "gender",
            "Geburtsjahresgruppe",
            "ageGroup",
        ]
    ].head(30)
)

Die folgenden Zeilen sind Bundes-, Geschlechts-, Alters- oder andere Summen. Sie werden angezeigt, bevor sie für die Detailverteilung entfernt werden.

In [ ]:
statistic_dimensions = ["state", "voteType", "gender", "ageGroup"]
summary_statistic_rows = normalized_state_statistics[
    ~normalized_state_statistics[statistic_dimensions].notna().all(axis=1)
]
print(f"Ausgeschlossene Summenzeilen: {len(summary_statistic_rows):,}")
display(
    summary_statistic_rows[
        ["Land", "Erst-/Zweitstimme", "Geschlecht", "Geburtsjahresgruppe"]
    ].head(50)
)

In [ ]:
state_statistic_details = select_state_statistic_detail_rows(
    normalized_state_statistics
)
print(f"Verwendete Detailzeilen: {len(state_statistic_details):,}")
display(state_statistic_details.head(20))

## 9. Statistik in lange Parteizeilen umformen

Die veröffentlichten absoluten Statistikwerte werden zunächst nur umgeformt. Noch werden sie nicht als amtliche Stimmenzahlen verwendet.

In [ ]:
state_statistic_votes = reshape_state_statistic_votes(state_statistic_details)
print(f"Statistikzellen: {len(state_statistic_votes):,}")
display(state_statistic_votes.head(30))

## 10. Gerundete Statistikwerte in Anteile umrechnen

Die repräsentative Statistik kann durch Rundung und Methodik vom amtlichen Endergebnis abweichen. Deshalb wird jede Zelle durch die statistische Summe ihrer Partei im jeweiligen Land und für die jeweilige Stimmenart geteilt.

Diese Anteile werden später mit den amtlichen Wahlkreis- und Wahlartsummen multipliziert. Die absoluten Statistikwerte werden also nicht als Endsumme übernommen.

In [ ]:
demographic_profiles = calculate_demographic_profiles(state_statistic_votes)

display(demographic_profiles.head(30))

share_checks = (
    demographic_profiles.groupby(["state", "voteType", "party"])["share"]
    .sum()
    .sub(1.0)
    .abs()
)
print(f"Größte Abweichung einer Anteils-Summe von 1: {share_checks.max():.12g}")

In [ ]:
# Beispiel: veröffentlichte statistische Summe und amtliche Summe nebeneinander.
statistic_totals = (
    demographic_profiles[["state", "voteType", "party", "statisticTotal"]]
    .drop_duplicates()
)
official_totals = (
    district_totals.groupby(["state", "voteType", "party"], as_index=False)["votes"]
    .sum()
    .rename(columns={"votes": "officialTotal"})
)
total_comparison = official_totals.merge(
    statistic_totals,
    on=["state", "voteType", "party"],
    how="inner",
)
total_comparison["difference"] = (
    total_comparison["officialTotal"] - total_comparison["statisticTotal"]
)
display(total_comparison.reindex(total_comparison["difference"].abs().sort_values(ascending=False).index).head(30))

## 11. Optionales Bundesmuster für Brief- und Urnenwahl

Diese dritte Datei ist nicht pro Wahlkreis. Sie enthält ein bundesweites Muster nach Partei, Geschlecht, Alter und Brief-/Urnenwahl. IPF benutzt es nur als Ausgangsmuster.

Ist kein Pfad eingetragen, wird ein neutrales Ausgangsmuster verwendet. Die bekannten Landesränder werden in beiden Fällen exakt angepasst.

In [ ]:
federal_method_seed = None

if FEDERAL_METHOD_DEMOGRAPHICS_CSV is None:
    print("Keine optionale Bundesdatei angegeben. Es wird ein neutrales Ausgangsmuster verwendet.")
else:
    raw_federal_method_statistics = read_local_csv(
        FEDERAL_METHOD_DEMOGRAPHICS_CSV,
        comment="#",
    )
    display(raw_federal_method_statistics.head(20))

    normalized_federal_method_statistics = normalize_federal_method_rows(
        raw_federal_method_statistics
    )
    display(normalized_federal_method_statistics.head(20))

    federal_method_details = select_federal_method_detail_rows(
        normalized_federal_method_statistics
    )
    federal_method_votes = reshape_federal_method_votes(federal_method_details)
    display(federal_method_votes.head(30))

    federal_method_seed = calculate_federal_method_weights(federal_method_votes)
    display(federal_method_seed.head(30))

## 12. Landesprofile mit IPF berechnen

Für jede Partei und Stimmenart werden zwei bekannte Ränder zusammengeführt:

- die demografischen Anteile aus der Landesstatistik;
- die amtlichen Brief- und Urnenstimmen aus der Wahlbezirksdatei.

Parteien ohne eigene Statistik verwenden das Profil von `Sonstige`. Nur wenn auch dieses fehlt, wird gleichverteilt.

In [ ]:
state_method_profiles = build_state_method_profiles(
    district_totals,
    demographic_profiles,
    federal_method_seed,
)

print(f"Profilzeilen: {len(state_method_profiles):,}")
display(
    state_method_profiles[
        ["party", "demographicProfileSource", "methodSeedSource"]
    ].drop_duplicates().sort_values(["demographicProfileSource", "party"])
)
display(state_method_profiles.head(30))

## 13. Landesprofile auf die amtlichen Wahlkreissummen anwenden

Nun gilt für jede Wahlkreis-Partei-Wahlart-Kombination:

`geschätzte Stimmen = amtliche Wahlkreissumme × berechneter Profilanteil`

Die Summe aller erzeugten Detailzeilen bleibt dabei gleich der amtlichen Ausgangssumme.

In [ ]:
first_district_totals = district_totals[district_totals["voteType"] == "1"].copy()
second_district_totals = district_totals[district_totals["voteType"] == "2"].copy()

first_votes = distribute_district_votes(first_district_totals, state_method_profiles)
second_votes = distribute_district_votes(second_district_totals, state_method_profiles)
all_votes = [*first_votes, *second_votes]

print(f"Erststimmen-Einträge: {len(first_votes):,}")
print(f"Zweitstimmen-Einträge: {len(second_votes):,}")
display(pd.DataFrame([asdict(entry) for entry in first_votes[:30]]))

## 14. Bekannte Summen prüfen

Die Validierung vergleicht die erzeugten Einträge wieder mit den amtlichen Wahlkreis-/Wahlartsummen und mit den berechneten demografischen Landesrändern.

In [ ]:
validation = validate_vote_entries(
    all_votes,
    district_totals,
    state_method_profiles,
)
validation

## 15. JSON-Dateien schreiben

Erst nach allen sichtbaren Prüfungen werden die beiden Dateien gespeichert.

In [ ]:
first_votes_path = write_vote_entries(
    first_votes,
    OUTPUT_DIRECTORY / "first_votes.json",
)
second_votes_path = write_vote_entries(
    second_votes,
    OUTPUT_DIRECTORY / "second_votes.json",
)

first_votes_path, second_votes_path